In [20]:
import pandas as pd
import spacy
import json
from spacy.language import Language
from spacy.tokens import Span
from spacy.training import Example
from spacy.scorer import Scorer
from thefuzz import fuzz, process
from tqdm.auto import tqdm
from newspaper import Article, Config
from newspaper.article import ArticleException
from selenium import webdriver
from selenium.webdriver import ChromeOptions
import threading
from dotenv import load_dotenv
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
tqdm.pandas()

In [21]:
load_dotenv()
serp_api_key = os.environ.get('SERP_API_KEY')

def get_selenium_html(url):
    with threading.Lock():
        with webdriver.Chrome(options=options) as driver: 
            driver.get(url)
            article_html = driver.page_source
        return article_html

options = ChromeOptions()
options.page_load_strategy = 'eager'
options.add_argument("--headless=new")

config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124  Safari/537.36'
config.request_timeout = 20

In [22]:
def parse_url(url) -> str | None:
    try:
        article = Article(url)
        article.download()
        article.parse()

        text = " ".join(article.text.split()[:200])
        if text == '':
            raise LookupError(f"First Attempt: Unable to extact article text for {url}. Retrying...") # if fail, go into second try

        return text
    
    except (ArticleException, LookupError) as e1:
        try:
            if str(e1).find('403') != -1 or isinstance(e1, LookupError):
                article = Article(url, config=config)
                article.download()
                article.parse()

                text = " ".join(article.text.split()[:200])
                if text == '':
                    raise LookupError(f"Second Attempt: Unable to extact article text for {url}. Retrying...") # if fail, go into third try
                else:
                    print(f'Retry succeeded for {url}!')
                
                return text
            
            else:
                print(e1)
                return ''
            
        except (ArticleException, LookupError) as e2:
            if str(e2).find('403') != -1 or isinstance(e2, LookupError):
                article = Article(url, config=config)
                article.download(input_html = get_selenium_html(url))
                article.parse()
                text = " ".join(article.text.split()[:200])
                if text == '':
                    text = ''
                    print(f"Final Attempt: Unable to extact article text for {url}") # if it still fails, can't circumvent.
                else:
                    print(f'Retry succeeded for {url}!')
                
                return text

            else:
                print(e2)
                return ''


def parse_stories_parallel(df, max_threads=80):

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        results = list(executor.map(parse_url, df['url']))

    df['text'] = results
    df['text'] = df['text'].replace('', pd.NA)
    
    return df

In [23]:
df_articles = pd.read_csv('elections_opinions.csv').sample(51, random_state=42)

df_articles

,id,indexed_date,language,media_name,media_url,publish_date,title,url
907,82c821900d8508795c49684a8c84141c2159f10d882bb6...,2024-05-04 12:30:11.715476+00:00,en,india.com,india.com,2022-08-02,Rishi Sunak tables radical tax vision as Liz T...,https://zeenews.india.com/world/rishi-sunak-ta...
617,70e8bf00c2be8de4372366237e1f63ddce65a714d2df13...,2024-08-17 04:18:32.944161+00:00,en,indianexpress.com,indianexpress.com,2024-08-17,Express View on announcement of J&K polls: Lon...,https://indianexpress.com/article/opinion/edit...
1386,f23852bcf2d7b82a230f75fbe8d1edbe26a3cebacb2bf7...,2024-02-17 16:42:27.034984+00:00,en,thequint.com,thequint.com,2023-11-04,Semi-final: The Big Difference Between BJP and...,https://www.thequint.com/opinion/elections-bjp...
941,51fbcaa3e505e641e2b538b19565494374ea19bbd779b6...,2024-05-01 02:59:14.266920+00:00,en,indiatoday.in,indiatoday.in,2022-09-28,"Sitaram Kesri, the Congress president who had ...",https://www.indiatoday.in/news-analysis/story/...
303,d6a52c534f692b225cc72ebaa4b4cd56934a7e13c72a34...,2024-11-20 11:45:53.726991+00:00,en,telegraphindia.com,telegraphindia.com,2019-05-25,What should worry us is the insidious recastin...,https://www.telegraphindia.com/opinion/what-sh...
1479,2e07ed30a7a6cdfd63b18753431d2c9c31831b48ca4d78...,2023-12-07 01:34:18+00:00,en,hindustantimes.com,hindustantimes.com,2023-12-05,Three lessons for the Congress party before 2024,https://www.hindustantimes.com/opinion/terms-o...
175,0b44d8ceae1c3f3dbb9c9257bb84cbed9548b3e63bd905...,2024-12-25 08:23:07.624172+00:00,en,indiatoday.in,indiatoday.in,2022-03-13,Why more women voted for the BJP in 2022 elect...,https://www.indiatoday.in/elections/story/why-...
342,d96751a6f667b22b5bd0e511e58f4e4892c861660eb7bc...,2024-11-09 02:52:05.528867+00:00,en,theprint.in,theprint.in,2020-05-20,The surveys can’t be true: Modi’s popularity h...,https://theprint.in/opinion/the-surveys-cant-b...
1308,65b152b6e1a574f2c7202bc484ea2385a7971d32d3b01c...,2024-02-24 21:00:56.619582+00:00,en,indiatimes.com,indiatimes.com,2023-07-10,BJP’s acquisition-and-alliance strategy is key...,https://economictimes.indiatimes.com/opinion/e...
1229,9d8c0fd395ea36a231afef57427b781c6a8fd51e47a608...,2024-02-29 23:33:44.445716+00:00,en,rediff.com,rediff.com,2023-04-14,Can Modi Swing It For BJP In Karnataka?,https://www.rediff.com/news/column/ramesh-meno...


In [24]:
df_articles = parse_stories_parallel(df_articles)
df_articles = df_articles.dropna(subset='text')
df_articles

Retry succeeded for https://www.hindustantimes.com/opinion/the-fine-print-of-karnataka-caste-census-101709905960498.html!
Retry succeeded for https://www.hindustantimes.com/opinion/keeping-up-with-up-modi-juggernaut-india-dreams-and-the-power-of-number-80-101710505811056.html!
Retry succeeded for https://www.hindustantimes.com/opinion/the-northeast-elections-hold-huge-significance-101671371364311.html!
Retry succeeded for https://www.hindustantimes.com/opinion/can-the-malady-of-urban-voter-apathy-be-remedied-101683636616738.html!
Retry succeeded for https://www.hindustantimes.com/opinion/verdict-2024-a-huge-asset-for-foreign-policy-101718115159858.html!
Retry succeeded for https://www.hindustantimes.com/opinion/needed-a-reframing-of-the-debate-on-jobs-101721143219042.html!
Retry succeeded for https://www.hindustantimes.com/opinion/terms-of-trade-three-lessons-for-the-congress-party-before-2024-101701764083801.html!
Retry succeeded for https://www.hindustantimes.com/india-news/lok-sabha

,id,indexed_date,language,media_name,media_url,publish_date,title,url,text
907,82c821900d8508795c49684a8c84141c2159f10d882bb6...,2024-05-04 12:30:11.715476+00:00,en,india.com,india.com,2022-08-02,Rishi Sunak tables radical tax vision as Liz T...,https://zeenews.india.com/world/rishi-sunak-ta...,London: As the voting process for the new Cons...
617,70e8bf00c2be8de4372366237e1f63ddce65a714d2df13...,2024-08-17 04:18:32.944161+00:00,en,indianexpress.com,indianexpress.com,2024-08-17,Express View on announcement of J&K polls: Lon...,https://indianexpress.com/article/opinion/edit...,In announcing the dates for the Assembly elect...
1386,f23852bcf2d7b82a230f75fbe8d1edbe26a3cebacb2bf7...,2024-02-17 16:42:27.034984+00:00,en,thequint.com,thequint.com,2023-11-04,Semi-final: The Big Difference Between BJP and...,https://www.thequint.com/opinion/elections-bjp...,The state assembly polls this month in five st...
941,51fbcaa3e505e641e2b538b19565494374ea19bbd779b6...,2024-05-01 02:59:14.266920+00:00,en,indiatoday.in,indiatoday.in,2022-09-28,"Sitaram Kesri, the Congress president who had ...",https://www.indiatoday.in/news-analysis/story/...,"The morning of March 14, 1998, was rather plea..."
303,d6a52c534f692b225cc72ebaa4b4cd56934a7e13c72a34...,2024-11-20 11:45:53.726991+00:00,en,telegraphindia.com,telegraphindia.com,2019-05-25,What should worry us is the insidious recastin...,https://www.telegraphindia.com/opinion/what-sh...,Every general election is distinctive and uniq...
1479,2e07ed30a7a6cdfd63b18753431d2c9c31831b48ca4d78...,2023-12-07 01:34:18+00:00,en,hindustantimes.com,hindustantimes.com,2023-12-05,Three lessons for the Congress party before 2024,https://www.hindustantimes.com/opinion/terms-o...,The Congress’s impressive Telangana victory in...
175,0b44d8ceae1c3f3dbb9c9257bb84cbed9548b3e63bd905...,2024-12-25 08:23:07.624172+00:00,en,indiatoday.in,indiatoday.in,2022-03-13,Why more women voted for the BJP in 2022 elect...,https://www.indiatoday.in/elections/story/why-...,The story of women voting in India has added o...
342,d96751a6f667b22b5bd0e511e58f4e4892c861660eb7bc...,2024-11-09 02:52:05.528867+00:00,en,theprint.in,theprint.in,2020-05-20,The surveys can’t be true: Modi’s popularity h...,https://theprint.in/opinion/the-surveys-cant-b...,It would be much more reliable to hear ground ...
1308,65b152b6e1a574f2c7202bc484ea2385a7971d32d3b01c...,2024-02-24 21:00:56.619582+00:00,en,indiatimes.com,indiatimes.com,2023-07-10,BJP’s acquisition-and-alliance strategy is key...,https://economictimes.indiatimes.com/opinion/e...,If BJP has to increase its electoral footprint...
1229,9d8c0fd395ea36a231afef57427b781c6a8fd51e47a608...,2024-02-29 23:33:44.445716+00:00,en,rediff.com,rediff.com,2023-04-14,Can Modi Swing It For BJP In Karnataka?,https://www.rediff.com/news/column/ramesh-meno...,This article was first published 2 years ago T...


In [25]:
df_corpus = pd.read_json('nivaduck_with_display_names.json')
df_corpus = df_corpus.dropna(subset='display_name')
corpus = df_corpus['display_name'].to_dict()

In [26]:
EST = {
    'BJP',
    'AIADMK',
    #'GOV',
    'NCP',
    'ABVP',
    'TDP',
    'AMMK',
    'JDU',
    'RSS',
    'JDS',
    'MNS',
    'LJP',
    'VHP'
}

OPP = {
    'INC',
    'DMK',
    'AAP',
    'SP',
    'SAMAJWADI',
    'SAMAJWADI ',
    'BJD',
    'TRS',
    'CPIM',
    'AITC',
    'TMC',
    'RJD',
    'AIMIM',
    'JKNC',
    'BSP',
    'JKPDP',
    'JMM',
    'CPIML'
}

In [134]:
nlp = spacy.load('en_core_web_trf')
@Language.component("fuzzy_affiliation")
def fuzzy_affiliates(doc):
    ents = []
    
    for ent in doc.ents:
        '''if ent.label_ not in {"PERSON", "ORG"} or ent.text in {'Lok Sabha', 'Assembly'}:
            ents.append(ent)
            continue
        
        if 'congress' in ent.text.lower(): match = process.extractOne(ent.text, corpus, scorer=fuzz.ratio, score_cutoff=95)
        else: match = process.extractOne(ent.text, corpus, scorer=fuzz.token_set_ratio, score_cutoff=95)'''
        
        if ent.label_ not in {"PERSON", "ORG"}:
            ents.append(ent)
            continue

        match = process.extractOne(ent.text, corpus, scorer=fuzz.ratio, score_cutoff=95)

        if not match:
            ents.append(ent)
            continue

        _, _, index =  match
        party = str(df_corpus.loc[index, 'party'])

        if party in EST:
            aff = 'EST'
        elif party in OPP:
            aff = 'OPP'
        else:
            ents.append(ent)
            continue
        
        new_ent = Span(doc, ent.start, ent.end, label=aff)
        ents.append(new_ent)
            
    doc.ents = ents
    return doc

nlp.add_pipe("fuzzy_affiliation", after='ner')

<function __main__.fuzzy_affiliates(doc)>

In [135]:
texts = list(df_articles['text'])

texts[:5]

['London: As the voting process for the new Conservative Party leader formally opened with postal ballots being mailed out to Tory members from Monday, Rishi Sunak vowed to cut the basic rate of income tax by 20 per cent in a few years if he is elected Britain\'s Prime Minister. The 42-year-old former Chancellor, who is trailing his rival Liz Truss in the opinion polls and bookmaker\'s odds, said his "radical vision" would mean a 20 per cent tax reduction, the "largest cut to income tax in 30 years". Tax cuts have become the dominant issue in the election campaign and while Truss has pledged cuts from day one, Sunak has sought to focus on a more measured approach in order to curb soaring inflation. "What I\'m putting to people today is a vision to deliver the biggest income tax cut since (former Tory Prime Minister) Margaret Thatcher\'s government," said Sunak. "It is a radical vision but it is also a realistic one and there are some core principles that I\'m simply not prepared to com

In [136]:
def doc_to_spans(doc):
    results = []
    for ent in doc.ents:
        if ent.label_ in {"EST", "OPP"}:
            results.append({
                'from_name': 'label',
                'to_name': 'text',
                'type': 'labels',
                'value': {
                    'start': ent.start_char,
                    'end': ent.end_char,
                    'text': ent.text,
                    'labels': [ent.label_]
                }
            })
    return results

tasks = []
for text in tqdm(texts):
    doc = nlp(text)
    spans = doc_to_spans(doc)
    tasks.append({
        'data': {'text': text},
        'predictions': [{'model_version': 'spacy_fuzzy_parties', 'result': spans}]
    })

with open('tasks_final.json', 'w') as f:
    json.dump(tasks, f, indent=2)

  0%|          | 0/50 [00:00<?, ?it/s]

In [137]:
with open('tasks_final.json', 'r') as f:
    data = json.load(f)

extracted_data = []
for task in data:
    for prediction in task.get("predictions", []):
        for result in prediction.get("result", []):
            value = result.get("value", {})
            
            text = value.get("text")
            labels = value.get("labels", [])
            label = labels[0] if labels else "Unknown"
            
            if text:
                extracted_data.append({"name": text, "party": label})

In [138]:
df_count = pd.DataFrame(extracted_data)
df_count = df_count.groupby(['name', 'party']).size().reset_index(name='freq')
df_count = df_count.sort_values(by='freq', ascending=False).reset_index(drop=True)
df_count

,name,party,freq
0,BJP,EST,75
1,Congress,OPP,40
2,Narendra Modi,EST,15
3,Rahul Gandhi,OPP,11
4,AAP,OPP,10
5,Ajit Pawar,EST,5
6,Mamata Banerjee,OPP,4
7,Sharad Pawar,EST,3
8,Nitish Kumar,EST,3
9,AICC,OPP,3


In [139]:
with open("ground_truth_final.json", "r", encoding="utf-8") as f:
    data = json.load(f)

examples = []
skipped_count = 0
for entry in tqdm(data):
    text = entry['text']
    labels = entry.get('label', [])
    
    pred_doc = nlp(text)
    filtered_preds = [ent for ent in pred_doc.ents if ent.label_ in {"EST", "OPP"}]
    pred_doc.ents = filtered_preds
    
    target_doc = nlp.make_doc(text) 
    # make target document object using same tokenization as predictions
    target_ents = []
    
    for label in labels: # labels from the ground truth
        start = label['start']
        end = label['end']
        label_name = label['labels'][0]
        
        span = target_doc.char_span(start, end, label=label_name, alignment_mode="strict")
        if span is None: span = target_doc.char_span(start, end, label=label_name, alignment_mode="contract")
        if span is None: span = target_doc.char_span(start, end, label=label_name, alignment_mode="expand")
            
        if span is None:
            print(f"Warning: Failed to align exact span for {label_name} at {start}-{end}")
            continue
            
        target_ents.append(span)

    
    target_doc.ents = target_ents
    example = Example(pred_doc, target_doc)
    examples.append(example)

  0%|          | 0/50 [00:00<?, ?it/s]

In [140]:
scorer = Scorer()
scores = scorer.score(examples)

print(f"Precision:\n{scores['ents_p']:.4f}\n")
print(f"Recall:\n{scores['ents_r']:.4f}\n")
print(f"F1 Score:\n{scores['ents_f']:.4f}")

Precision:
0.9860

Recall:
0.5845

F1 Score:
0.7339
